# Archived: original Colab notebook (May 2025)

This is the notebook submitted with the BSc thesis, kept for reference with outputs cleared. **The maintained code is the `bookscreener` package and `scripts/` in this repository.**

Known issues in this version, fixed in the refactor:

1. The Gradio app loads `/content/model`, which cell 21 overwrites with a 3-row demo CatBoost model; the trained meta-model was never saved.
2. CatBoost is trained on 6 features (incl. `atesman_score`) but the app passes 5.
3. CatBoost is fitted on BERT's predictions for BERT's own training data, so the stack cannot improve on BERT (test F1 0.924 vs 0.926 for BERT alone).
4. Category explanations use substring matching (`'am' in text`), so almost every Turkish text matches.
5. `str.lower()` mishandles Turkish `I`/`İ`; the Ateşman score is computed on mixed-case text.
6. Hard-coded Google Drive paths; the validation split is not used during BERT training.


In [ ]:
!pip install -q transformers datasets catboost optuna scikit-learn

import pandas as pd
from sklearn.model_selection import train_test_split
from google.colab import drive

drive.mount('/content/drive')

# CSV dosyalarının yolunu tanımlama kısmı
train_path = "/content/drive/MyDrive/proje_dizini/Twitter/train.csv"
valid_path = "/content/drive/MyDrive/proje_dizini/Twitter/valid.csv"
test_path  = "/content/drive/MyDrive/proje_dizini/Twitter/test.csv"

# CSV dosyalarını pandas ile okutma
train_df = pd.read_csv(train_path)
valid_df = pd.read_csv(valid_path)
test_df  = pd.read_csv(test_path)

# Veri setlerinin boyutlarını kontrol etme
print("Train set boyutu:", train_df.shape)
print("Validation set boyutu:", valid_df.shape)
print("Test set boyutu:", test_df.shape)

train_df.head(5)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Olası Hata Durumunda Çalıştır

!pip uninstall -y numpy
!pip install numpy==1.26.4

In [ ]:
import re

# Temizleme için fonksiyon
def clean_tweet(text):
    text = str(text)

    text = re.sub(r"@[\w_]+", "<USER>", text)
    # URL'leri temizleme
    text = re.sub(r"http\S+|www\.\S+", "<URL>", text)
    # Hashtag işaretlerini kaldırma, Emoji ve özel karakterleri opsiyonel olarak kaldırma
    text = re.sub(r"#", "", text)

    text = re.sub(r"[^\w\s,.!?<>'\"-]", "", text)
    # Fazla boşlukları sadeleştirme
    text = re.sub(r"\s+", " ", text).strip()
    # Küçük harfe çevirme
    text = text.lower()
    return text

# Eğitim, doğrulama ve test metinlerine temizleme
train_df['text_clean'] = train_df['text'].apply(clean_tweet)
valid_df['text_clean'] = valid_df['text'].apply(clean_tweet)
test_df['text_clean']  = test_df['text'].apply(clean_tweet)

print("Temizlenmiş örnek tweet:")
print("Öncesi:", train_df['text'].iloc[0])
print("Sonrası:", train_df['text_clean'].iloc[0])


In [ ]:
# Hece sayısı için fonksiyon
def count_syllables(word):
    vowels = "aeıioöuü"
    return sum(1 for char in word if char in vowels)

# Ateşman Okunabilirlik Skoru hesaplama fonksiyonu
def calculate_atesman_score(text):
    sentences = re.split(r'[.!?]', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    num_sentences = len(sentences)

    words = re.findall(r'\b\w+\b', text)
    num_words = len(words)
    num_syllables = sum(count_syllables(word) for word in words)

    if num_words == 0 or num_sentences == 0:
        return 0.0

    score = 198.825 - 40.175 * (num_syllables / num_words) - 2.610 * (num_words / num_sentences)
    return round(score, 2)


In [ ]:
# Uygunsuz kelime listesini okuma
lexicon_path = "/content/drive/MyDrive/proje_dizini/uygunsuzkelimeler/ofansif.txt"
with open(lexicon_path, "r", encoding="utf-8") as f:
    offensive_words = {line.strip().lower() for line in f if line.strip()}  # Küçük harfe çevirme ve set yaptığımız kısım

print(f"Uygunsuz kelime listesinde {len(offensive_words)} adet ifade yüklendi.")

sample_words = list(offensive_words)[:10]
print("Örnek uygunsuz kelimeler:", sample_words)


In [ ]:
# Tweet'te geçen uygunsuz kelime sayısını bulan fonksiyon
def count_offensive_words(text):
    words = text.split()  # temizlenmiş metni boşluklardan ayır
    count = sum(1 for w in words if w in offensive_words)
    return count

# Eğitim, doğrulama, test veri çerçevelerine uygulama
train_df['offensive_count'] = train_df['text_clean'].apply(count_offensive_words)
valid_df['offensive_count'] = valid_df['text_clean'].apply(count_offensive_words)
test_df['offensive_count']  = test_df['text_clean'].apply(count_offensive_words)

#uygunsuz kelime var mı yok mu kısmı
train_df['has_offensive'] = (train_df['offensive_count'] > 0).astype(int)
valid_df['has_offensive'] = (valid_df['offensive_count'] > 0).astype(int)
test_df['has_offensive']  = (test_df['offensive_count'] > 0).astype(int)

# Uygunsuz kelime oranı hesaplama : (offensive_ratio) = offensive_count / total word count
train_df['offensive_ratio'] = train_df['offensive_count'] / (train_df['text_clean'].str.split().str.len() + 1e-9)
valid_df['offensive_ratio'] = valid_df['offensive_count'] / (valid_df['text_clean'].str.split().str.len() + 1e-9)
test_df['offensive_ratio']  = test_df['offensive_count']  / (test_df['text_clean'].str.split().str.len()  + 1e-9)

# Ateşman okunabilirlik skorlarını hesaplama ve ekleme
train_df["atesman_score"] = train_df["text_clean"].apply(calculate_atesman_score)
valid_df["atesman_score"] = valid_df["text_clean"].apply(calculate_atesman_score)
test_df["atesman_score"]  = test_df["text_clean"].apply(calculate_atesman_score)

example_idx = 0
print("Örnek Tweet:", train_df.loc[example_idx, 'text_clean'])
print("Uygunsuz kelime sayısı:", train_df.loc[example_idx, 'offensive_count'])
print("Uygunsuz kelime var mı?:", train_df.loc[example_idx, 'has_offensive'])
print("Uygunsuz kelime oranı:", train_df.loc[example_idx, 'offensive_ratio'])


In [ ]:
import torch
from transformers import BertTokenizerFast, BertForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from datasets import Dataset

In [ ]:
import torch
from transformers import BertTokenizerFast, BertForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from datasets import Dataset

# Model ve tokenizer'ı yükleme
model_name = "dbmdz/bert-base-turkish-128k-uncased"
tokenizer = BertTokenizerFast.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)

# GPU kontrolü(colab bazen cpu da kullanmamızı mecbur kıldığında kullanmak için)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Train ve validation verilerini HuggingFace Dataset formatına çevirme
train_dataset = Dataset.from_pandas(train_df[['text_clean', 'label']].reset_index(drop=True))
valid_dataset = Dataset.from_pandas(valid_df[['text_clean', 'label']].reset_index(drop=True))

# Tokenize etme fonksiyonu
def tokenize_function(examples):
    return tokenizer(examples['text_clean'], padding="max_length", truncation=True, max_length=128)

# Tokenize edilmiş verisetlerini oluşturma
train_dataset = train_dataset.map(tokenize_function, batched=True)
valid_dataset = valid_dataset.map(tokenize_function, batched=True)

# Gereksiz sütunları kaldırma
train_dataset = train_dataset.remove_columns(['text_clean'])
valid_dataset = valid_dataset.remove_columns(['text_clean'])

# "label" sütununu Trainer'ın beklediği formatta "labels" olarak yeniden adlandırma
train_dataset = train_dataset.rename_column("label", "labels")
valid_dataset = valid_dataset.rename_column("label", "labels")

# Tensor formatına geçtiğimiz kısım (Trainer ve PyTorch için gerekli)
train_dataset.set_format("torch")
valid_dataset.set_format("torch")

# Değerlendirme metriklerini hesaplama fonksiyonu
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=1)
    acc = accuracy_score(labels, predictions)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

training_args = TrainingArguments(
    output_dir="outputs",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    logging_dir="logs"
)

# Trainer oluşturma
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics
)

# Eğitim
trainer.train()


In [ ]:
# Eğitilen modeli kaydetme (model + ağırlıklar + config)
trainer.save_model("/content/drive/MyDrive/CBAnalysis/model")

# Tokenizer'ı kaydetme
tokenizer.save_pretrained("/content/drive/MyDrive/CBAnalysis/tokenizer")


In [ ]:
#Eğitilen Modeli Tekrar Çağırma Kodu

from transformers import BertTokenizerFast, BertForSequenceClassification

# Tokenizer'ı yükleme kodu
tokenizer = BertTokenizerFast.from_pretrained("/content/drive/MyDrive/CBAnalysis/tokenizer")

# Eğitilen modeli yükleme kodu
model = BertForSequenceClassification.from_pretrained("/content/drive/MyDrive/CBAnalysis/model")







In [ ]:
#Cihaza Atama kodu colab

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()  # Test/analiz moduna al

In [ ]:
import numpy as np
import torch

# BERT modelinden batch şeklinde çıktı olasılıkları ve tahminleri alan fonksiyon
def get_bert_prediction_prob_batched(text_list, batch_size=64):
    model.eval()
    offensive_probs = []
    preds = []

    for i in range(0, len(text_list), batch_size):
        batch_texts = text_list[i:i+batch_size]
        encodings = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors='pt')
        encodings = {k: v.to(device) for k, v in encodings.items()}

        with torch.no_grad():
            outputs = model(**encodings)
            logits = outputs.logits
            probs = torch.nn.functional.softmax(logits, dim=-1)
            offensive_probs.extend(probs[:, 1].cpu().numpy())
            preds.extend(logits.argmax(dim=-1).cpu().numpy())

        # Bellek temizliği kodu
        del encodings, outputs, logits, probs
        torch.cuda.empty_cache()

    return offensive_probs, preds


In [ ]:
# Eğitim setinde tahminleri batch'li olarak alma
batch_size = 64  # GPU RAM'e göre ayarlanabilir, duruma göre buradan değiştiririz
bert_train_probs = []
bert_train_preds = []

for i in range(0, len(train_df), batch_size):
    batch_texts = train_df['text_clean'].iloc[i:i+batch_size].tolist()
    probs, preds = get_bert_prediction_prob_batched(batch_texts, batch_size=batch_size)
    bert_train_probs.extend(probs)
    bert_train_preds.extend(preds)

# Eğitim setine yeni sütunlar ekleme
train_df['bert_offensive_prob'] = pd.Series(bert_train_probs, index=train_df.index)
train_df['bert_pred'] = pd.Series(bert_train_preds, index=train_df.index)


In [ ]:
# Validation ve test seti için tahminleri batch'li şekilde alma
for df, name in [(valid_df, "Validation"), (test_df, "Test")]:
    print(f"{name} seti için tahmin başlatıldı...")
    probs, preds = get_bert_prediction_prob_batched(df['text_clean'].tolist(), batch_size=64)
    df['bert_offensive_prob'] = probs
    df['bert_pred'] = preds
    print(f"{name} seti - Pozitif (ofansif) sınıf oranı: {df['bert_pred'].mean():.2f}")


In [ ]:
# Ateşman ile ilgili ekstra denetleme
test_df["atesman_score"] = test_df["text_clean"].apply(calculate_atesman_score)


In [ ]:
from catboost import CatBoostClassifier, Pool

# Eğitim için özellik ve hedefi ayarlama
feature_cols = ['offensive_count', 'has_offensive', 'offensive_ratio', 'bert_offensive_prob', 'bert_pred', 'atesman_score']

X_train = train_df[feature_cols]
y_train = train_df['label']
X_valid = valid_df[feature_cols]
y_valid = valid_df['label']
X_test  = test_df[feature_cols]
y_test  = test_df['label']

# CatBoost veri havuzları (Pool) oluşturma kısmı
train_pool = Pool(X_train, label=y_train)
valid_pool = Pool(X_valid, label=y_valid)


In [ ]:
import optuna

# Optuna amacı (objective) tanımlama
def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 300),
        "depth": trial.suggest_int("depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 10, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-2, 2, log=True),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "eval_metric": "F1",
        "use_best_model": True,
        "loss_function": "Logloss",
        "verbose": False
    }
    # CatBoostClassifier ile eğitim yaptığımız kısım
    model_cb = CatBoostClassifier(**params)
    model_cb.fit(train_pool, eval_set=valid_pool, early_stopping_rounds=20, verbose=False)
    # Doğrulama seti üzerinde F1 skoru
    preds_valid = model_cb.predict(X_valid)
    _, _, f1, _ = precision_recall_fscore_support(y_valid, preds_valid, average='binary')
    return f1

# Optuna aramasını gerçekleştir (örnek 20 deneme ile sınırlı, istersek değiştirilebilir)
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print("En iyi parametreler:", study.best_params)
print("En iyi doğrulama F1 skoru:", study.best_value)


In [ ]:
# Optuna ile bulunan en iyi parametreler varsa onları kullan, yoksa elle belirle kodu
best_params = study.best_params if 'study' in locals() else {
    "iterations": 200,
    "depth": 6,
    "learning_rate": 0.1,
    "l2_leaf_reg": 3,
    "random_strength": 0.1,
    "border_count": 128
}

# En iyi parametrelerle model oluşturma
cat_model = CatBoostClassifier(
    **best_params,
    eval_metric="F1",
    use_best_model=True,
    loss_function="Logloss",
    random_seed=42
)

# Modeli eğitim verisiyle eğitme (doğrulama setini validasyon olarak veriyoruz)
cat_model.fit(train_pool, eval_set=valid_pool, early_stopping_rounds=20, verbose=50)


In [ ]:
# Test setinde tahmin yapma
test_preds = cat_model.predict(X_test)
test_probs = cat_model.predict_proba(X_test)[:, 1]  # ofansif olma ihtimali (2. sütun)

# Değerlendirme metriklerini hesaplama
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

acc = accuracy_score(y_test, test_preds)
prec = precision_score(y_test, test_preds)
rec = recall_score(y_test, test_preds)
f1 = f1_score(y_test, test_preds)
cm = confusion_matrix(y_test, test_preds)

print("Test Accuracy: {:.4f}".format(acc))
print("Test Precision: {:.4f}".format(prec))
print("Test Recall: {:.4f}".format(rec))
print("Test F1-Score: {:.4f}".format(f1))
print("Confusion Matrix:\n", cm)


In [ ]:
!pip install gradio

In [ ]:
from catboost import CatBoostClassifier, Pool

train_data = [[1, 3],
              [0, 4],
              [1, 7]]
train_labels = [1, 0, 1]

model = CatBoostClassifier(learning_rate=0.03)
model.fit(train_data,
          train_labels,
          verbose=False)

model.save_model("model")

from_file = CatBoostClassifier()

from_file.load_model("model")



In [ ]:
import torch
import re
import numpy as np
import pandas as pd
from transformers import BertTokenizerFast, BertForSequenceClassification
from catboost import CatBoostClassifier
import gradio as gr

# NLTK ve cümle ayırıcıyı indirme
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

# MODELLERİ YÜKLEME
# Tokenizer ve BERT model
tokenizer = BertTokenizerFast.from_pretrained("/content/drive/MyDrive/CBAnalysis/tokenizer")
bert_model = BertForSequenceClassification.from_pretrained("/content/drive/MyDrive/CBAnalysis/model").to("cpu").eval()

# CatBoost modeli
cat_model = CatBoostClassifier()
cat_model.load_model("/content/model")  # buraya model dosyanın doğru yolunu koymalısın

# Ofansif kelime listesi
with open("/content/drive/MyDrive/proje_dizini/uygunsuzkelimeler/ofansif.txt", "r", encoding="utf-8") as f:
    offensive_words = [w.strip().lower() for w in f.readlines()]


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
import zipfile
import os

# Hedef klasörü oluşturma
nltk_target_path = "/content/nltk_data/tokenizers"
os.makedirs(nltk_target_path, exist_ok=True)

# Zip’i açma
with zipfile.ZipFile("punkt.zip", 'r') as zip_ref:
    zip_ref.extractall(nltk_target_path)


In [ ]:
import nltk
nltk.data.path.append("/content/nltk_data")


In [ ]:
import pickle

with open("/content/nltk_data/tokenizers/punkt/turkish.pickle", "rb") as f:
    tokenizer = pickle.load(f)

# Artık sent_tokenize yerine şunu kullanalım
text = "Merhaba çocuklar. Bugün size masal anlatacağım! Hazır mısınız?"
sentences = tokenizer.tokenize(text)
print(sentences)


In [ ]:
import gradio as gr
import pandas as pd
import torch
import re
import os
import pickle
from transformers import BertTokenizerFast, BertForSequenceClassification
from catboost import CatBoostClassifier


In [ ]:
# BERT Model
tokenizer = BertTokenizerFast.from_pretrained("/content/drive/MyDrive/CBAnalysis/tokenizer")
bert_model = BertForSequenceClassification.from_pretrained("/content/drive/MyDrive/CBAnalysis/model").to("cpu").eval()

# CatBoost
cat_model = CatBoostClassifier()
cat_model.load_model("/content/model")

# Ofansif kelime listesi
with open("/content/drive/MyDrive/proje_dizini/uygunsuzkelimeler/ofansif.txt", "r", encoding="utf-8") as f:
    offensive_words = [w.strip().lower() for w in f.readlines()]


In [ ]:
!pip install PyMuPDF


In [ ]:
import fitz  # PyMuPDF için(pdf okutmak için lazım)

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = ""
    for page in doc:
        full_text += page.get_text()
    doc.close()
    return full_text


In [ ]:
def analyze_from_pdf(file):
    try:
        text = extract_text_from_pdf(file.name)
        return analyze_book_text(text)
    except Exception as e:
        return f"Hata: {str(e)}", ""


In [ ]:
kitap_dizini = "/content/drive/MyDrive/ornek_pdf"  # Burayı düzenleriz(hazır kitaplar database i için düşünüldü, uzun vadede belki gerekir şimdilik kullanmıyoruz)

def analyze_book_text(text):
    sentences = tokenizer_tr.tokenize(text)
    if len(sentences) == 0:
        return "Hiç cümle bulunamadı.", ""

    risky_sentences = []
    total = 0
    risky = 0

    for sentence in sentences:
        text_clean = re.sub(r"[^a-zA-ZçğıöşüÇĞİÖŞÜ0-9\s]", "", sentence.lower())
        tokens = text_clean.split()
        offensive_count = sum([1 for t in tokens if t in offensive_words])
        has_offensive = int(offensive_count > 0)
        offensive_ratio = offensive_count / len(tokens) if tokens else 0

        inputs = tokenizer(text_clean, return_tensors="pt", padding=True, truncation=True, max_length=128)
        with torch.no_grad():
            outputs = bert_model(**inputs)
            logits = outputs.logits
            probs = torch.nn.functional.softmax(logits, dim=-1)
            bert_offensive_prob = probs[:, 1].item()
            bert_pred = int(torch.argmax(logits, dim=-1))

        df_features = pd.DataFrame([{
            "offensive_count": offensive_count,
            "has_offensive": has_offensive,
            "offensive_ratio": offensive_ratio,
            "bert_offensive_prob": bert_offensive_prob,
            "bert_pred": bert_pred
        }])
        final_pred = int(cat_model.predict(df_features)[0])

        total += 1
        if final_pred == 1:
            risky += 1
            risky_sentences.append(sentence)

    risk_ratio = risky / total
    verdict = "Uygun ✅" if risk_ratio < 0.1 else "Uygun Değil ❌"
    summary = f"{total} cümleden {risky} tanesi riskli bulundu. (%{round(risk_ratio*100, 2)}) → {verdict}"

    örnekler = "\n\n".join(risky_sentences[:5]) if risky_sentences else "Riskli cümle bulunamadı."
    return summary, örnekler



In [ ]:
def analyze_from_drive(filename):
    full_path = os.path.join(kitap_dizini, filename)
    with open(full_path, "r", encoding="utf-8") as f:
        text = f.read()
    return analyze_book_text(text)


In [ ]:
def analyze_from_upload(file):
    with open(file.name, "r", encoding="utf-8") as f:
        text = f.read()
    return analyze_book_text(text)


In [ ]:
def count_syllables(word):
    vowels = "aeıioöuü"
    return sum(1 for char in word if char in vowels)

def calculate_atesman_score(text):
    sentences = re.split(r'[.!?]', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    num_sentences = len(sentences)

    words = re.findall(r'\b\w+\b', text)
    num_words = len(words)
    num_syllables = sum(count_syllables(word) for word in words)

    if num_words == 0 or num_sentences == 0:
        return 0.0

    score = 198.825 - 40.175 * (num_syllables / num_words) - 2.610 * (num_words / num_sentences)
    return round(score, 2)


In [ ]:
def analyze_book_text(text):
    sentences = tokenizer_tr.tokenize(text)
    if len(sentences) == 0:
        return "Hiç cümle bulunamadı.", ""

    risky_sentences = []
    total = 0
    risky = 0

    for sentence in sentences:
        text_clean = re.sub(r"[^a-zA-ZçğıöşüÇĞİÖŞÜ0-9\s]", "", sentence.lower())
        tokens = text_clean.split()
        offensive_count = sum([1 for t in tokens if t in offensive_words])
        has_offensive = int(offensive_count > 0)
        offensive_ratio = offensive_count / len(tokens) if tokens else 0

        inputs = tokenizer(text_clean, return_tensors="pt", padding=True, truncation=True, max_length=128)
        with torch.no_grad():
            outputs = bert_model(**inputs)
            logits = outputs.logits
            probs = torch.nn.functional.softmax(logits, dim=-1)
            bert_offensive_prob = probs[:, 1].item()
            bert_pred = int(torch.argmax(logits, dim=-1))

        df_features = pd.DataFrame([{
            "offensive_count": offensive_count,
            "has_offensive": has_offensive,
            "offensive_ratio": offensive_ratio,
            "bert_offensive_prob": bert_offensive_prob,
            "bert_pred": bert_pred
        }])
        final_pred = int(cat_model.predict(df_features)[0])

        total += 1
        if final_pred == 1:
            risky += 1
            risky_sentences.append(sentence)

    # ✔ Ateşman skoru burada hesaplanıyor
    atesman_score = calculate_atesman_score(text)

    risk_ratio = risky / total
    verdict = "Uygun ✅" if risk_ratio < 0.1 else "Uygun Değil ❌"
    summary = f"{total} cümleden {risky} tanesi riskli bulundu. (%{round(risk_ratio*100, 2)})\n"
    summary += f"Ateşman Okunabilirlik Skoru: {atesman_score} → "
    summary += "Kolay Okunabilir 📘" if atesman_score > 60 else "Zorlayıcı Metin 📕"
    summary += f"\n\nSonuç: {verdict}"

    örnekler = "\n\n".join(risky_sentences[:5]) if risky_sentences else "Riskli cümle bulunamadı."
    return summary, örnekler


In [ ]:
!pip install reportlab matplotlib


In [ ]:
!pip install -q gradio transformers catboost reportlab pymupdf nltk

import os
import re
import gradio as gr
import torch
import pickle
import fitz  # PyMuPDF unutma!
import pandas as pd
import matplotlib.pyplot as plt
from io import BytesIO
from transformers import BertTokenizerFast, BertForSequenceClassification
from catboost import CatBoostClassifier
from nltk.tokenize.punkt import PunktSentenceTokenizer
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4

# Model ve kaynakları yükleme
with open("/content/nltk_data/tokenizers/punkt/turkish.pickle", "rb") as f:
    tokenizer_tr: PunktSentenceTokenizer = pickle.load(f)

tokenizer = BertTokenizerFast.from_pretrained("/content/drive/MyDrive/CBAnalysis/tokenizer")
bert_model = BertForSequenceClassification.from_pretrained("/content/drive/MyDrive/CBAnalysis/model", trust_remote_code=True).to("cpu").eval()

cat_model = CatBoostClassifier()
cat_model.load_model("/content/model")

with open("/content/drive/MyDrive/proje_dizini/uygunsuzkelimeler/ofansif.txt", "r", encoding="utf-8") as f:
    offensive_words = [w.strip().lower() for w in f.readlines()]

# Ateşman skoru hesaplama
def count_syllables(word): return sum(1 for char in word if char in "aeıioöuü")
def calculate_atesman_score(text):
    sentences = re.split(r'[.!?]', text)
    words = re.findall(r'\b\w+\b', text)
    if not words or not sentences: return 0.0
    return round(198.825 - 40.175 * (sum(count_syllables(w) for w in words) / len(words)) - 2.610 * (len(words) / len(sentences)), 2)

# PDF'ten metin çıkarma
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ''.join([page.get_text() for page in doc])
    doc.close()
    return text

# Gerekçeli açıklama üretme kısmı
def generate_reasoned_explanation(text, verdict):
    if "Uygun ✅" in verdict:
        return ""

    kategori_dict = {
        "Şiddet ve Savaş": ["savaş", "katliam", "terör", "bombalamak", "silah", "intihar", "darbe", "vurmak", "cephane", "asker", "komando", "komutan", "hain"],
        "Uyuşturucu ve Kötü Alışkanlıklar": ["sigara", "esrar", "uyuşturucu", "kokain", "eroin", "torbacı", "alkol"],
        "Cinsellik ve Müstehcenlik": ["seks", "vajina", "sik", "am", "göt", "domalmak", "pezevenk", "orospu", "piç"],
        "Din ve Irk Ayrımcılığı": ["yahudi", "ermeni", "zenci", "ateist", "gavur", "siyonist", "kürt", "çingene"],
        "Aşağılama / Küfür / Argo": ["lan", "ulan", "şerefsiz", "salak", "aptal", "taşak", "bok", "kolpa", "puşt"],
        "Psikolojik / Ruhsal Sağlık": ["hasta", "manyak", "delirmek", "bunalım", "kriz", "meczup", "yobaz"],
        "Yasa Dışı ve Suç Unsuru": ["hırsız", "tecavüz", "taciz", "gasp", "katil", "şantaj", "dolandırıcı"],
        "Diğer Uygunsuzluklar": ["ahlaksız", "görgüsüz", "zındık", "çirkef", "pis", "kolpacı", "fahişe"]
    }

    kategori_aciklamalari = {
        "Şiddet ve Savaş": "şiddet, savaş veya terör içeriği barındırdığı için",
        "Uyuşturucu ve Kötü Alışkanlıklar": "uyuşturucu veya kötü alışkanlıkları teşvik edebileceği için",
        "Cinsellik ve Müstehcenlik": "çocuklar için uygun olmayan cinsel veya müstehcen içerikler barındırdığı için",
        "Din ve Irk Ayrımcılığı": "ayrımcılığa veya nefret söylemine yol açabileceği için",
        "Aşağılama / Küfür / Argo": "küfür, argo ya da hakaret içerdiği için",
        "Psikolojik / Ruhsal Sağlık": "psikolojik açıdan rahatsız edici ifadeler içerdiği için",
        "Yasa Dışı ve Suç Unsuru": "yasadışı ya da suça yönlendirebilecek içerikler barındırdığı için",
        "Diğer Uygunsuzluklar": "toplumsal olarak uygun görülmeyen içerikler barındırdığı için"
    }

    metin_l = text.lower()
    bulunan_kategoriler = set()
    for kategori, kelimeler in kategori_dict.items():
        for kelime in kelimeler:
            if kelime in metin_l:
                bulunan_kategoriler.add(kategori)
                break

    if not bulunan_kategoriler:
        return "❌ Uygunsuz içerik tespit edilemedi ancak model bu metni riskli olarak değerlendirdi."

    return "\n".join(f"- {kategori_aciklamalari[k]}" for k in bulunan_kategoriler)

# Analiz fonksiyonu
def analyze_book_text_with_charts(file):
    # Dosyayı okuma kısmı
    if file.name.endswith(".pdf"):
        text = extract_text_from_pdf(file.name)
    else:
        with open(file.name, "r", encoding="utf-8") as f:
            text = f.read()

    sentences = tokenizer_tr.tokenize(text)
    risky_sentences, bert_probs = [], []
    total, risky = 0, 0

    for sentence in sentences:
        text_clean = re.sub(r"[^a-zA-ZçğıöşüÇĞİÖŞÜ0-9\s]", "", sentence.lower())
        tokens = text_clean.split()
        offensive_count = sum(1 for t in tokens if t in offensive_words)
        features = pd.DataFrame([{
            "offensive_count": offensive_count,
            "has_offensive": int(offensive_count > 0),
            "offensive_ratio": offensive_count / len(tokens) if tokens else 0,
            "bert_offensive_prob": 0,
            "bert_pred": 0
        }])

        # BERT tahmini yaptığımız kısım
        inputs = tokenizer(text_clean, return_tensors="pt", padding=True, truncation=True, max_length=128)
        with torch.no_grad():
            logits = bert_model(**inputs).logits
            probs = torch.nn.functional.softmax(logits, dim=-1)
            features["bert_offensive_prob"] = probs[:, 1].item()
            features["bert_pred"] = int(torch.argmax(logits, dim=-1))
            bert_probs.append(probs[:, 1].item())

        pred = int(cat_model.predict(features)[0])
        total += 1
        if pred == 1:
            risky += 1
            risky_sentences.append(sentence)

    risk_ratio = risky / total
    atesman = calculate_atesman_score(text)
    verdict = "Uygun ✅" if risk_ratio < 0.03 else "Uygun Değil ❌"

    #  Pie Chart kodu
    fig1, ax1 = plt.subplots()
    ax1.pie([risky, total - risky], labels=['Riskli', 'Güvenli'], autopct='%1.1f%%', colors=["red", "green"])
    ax1.set_title("Cümle Bazlı Risk Oranı")
    pie_path = "/tmp/pie_chart.png"
    plt.savefig(pie_path)
    plt.close()

    #  Histogram kodu
    fig2, ax2 = plt.subplots()
    ax2.hist(bert_probs, bins=20, color="skyblue", edgecolor="black")
    ax2.set_title("Modelin Güven Dağılımı")
    ax2.set_xlabel("Ofansif Olma Olasılığı")
    ax2.set_ylabel("Cümle Sayısı")
    hist_path = "/tmp/hist_chart.png"
    plt.savefig(hist_path)
    plt.close()

    #  PDF Oluşturma kodu
    pdf_path = "/tmp/analiz_raporu.pdf"
    c = canvas.Canvas(pdf_path, pagesize=A4)
    c.setFont("Helvetica", 12)
    lines = [
        f"Toplam Cümle: {total}",
        f"Riskli Cümle Sayısı: {risky}",
        f"Risk Oranı: %{round(risk_ratio * 100, 2)}",
        f"Ateşman Skoru: {atesman}",
        f"Uygunluk Kararı: {verdict}",
        "",
        "Riskli Cümleler (İlk 5):"
    ] + risky_sentences[:5]

    y = 800
    for line in lines:
        for sub in re.findall(r'.{1,95}', line):
            c.drawString(50, y, sub)
            y -= 18
            if y < 50:
                c.showPage()
                y = 800
    c.save()

    # Sonuç yazısı + açıklama kodu
    summary = f"""
📊 **İstatistiksel Özet**
• Cümle sayısı: {total}
• Riskli cümle: {risky} (%{round(risk_ratio * 100, 2)})
• Ateşman Skoru: {atesman} → {"Kolay Okunabilir 📘" if atesman > 60 else "Zorlayıcı 📕"}
• Nihai Karar: {verdict}
    """.strip()

    gerekce = generate_reasoned_explanation(text, verdict)
    if gerekce:
        summary += "\n\n🧠 Gerekçeli Açıklama:\n" + gerekce

    examples = "\n\n".join(risky_sentences[:5]) or "Riskli cümle bulunamadı."

    return summary, examples, pie_path, hist_path, pdf_path

# Gradio Arayüz Tasarımı Kısmı
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("## 📚 Çocuk Kitabı Analiz Sistemi")
    gr.Markdown("KİTAPLARINIZI YÜKLEYEREK ÇOCUĞUNUZ İÇİN UYGUN OLUP OLMADIĞINI HIZLICA ÖĞRENEBİLİRSİNİZ")

    with gr.Row():
        file_input = gr.File(label="📂 Kitap Dosyası (.txt veya .pdf)")
        btn = gr.Button("🚀 Analizi Başlat")

    with gr.Row():
        out1 = gr.Textbox(label="📊 Özet", lines=7, interactive=False)
        out2 = gr.Textbox(label="⚠️ Riskli Cümleler", lines=7, interactive=False)

    with gr.Row():
        chart1 = gr.Image(label="🍕 Risk Oranı")
        chart2 = gr.Image(label="📈 Model Güven Dağılımı")

    with gr.Row():
        pdf_out = gr.File(label="📄 PDF Rapor (İndir)", interactive=False)

    gr.Markdown("---")
    gr.Markdown("👨‍💻 Hazırlayanlar: **Talha Çağrı Terzioğlu ve Mehmet Emre Kesekoğlu**  \n📘 Bitirme Tezi – Çocuk Kitaplarının Uygunluk Analizi")

    btn.click(fn=analyze_book_text_with_charts,
              inputs=file_input,
              outputs=[out1, out2, chart1, chart2, pdf_out])

demo.launch(debug=True)


In [ ]:
# Model Başarı Testleri

import torch
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from transformers import BertTokenizerFast, BertForSequenceClassification
from torch.utils.data import DataLoader, TensorDataset

# Model ve Tokenizer yolları
model_path = "/content/drive/MyDrive/CBAnalysis/model"
tokenizer_path = "/content/drive/MyDrive/CBAnalysis/tokenizer"

# Modeli safetensors formatında yükleme
model = BertForSequenceClassification.from_pretrained(model_path, trust_remote_code=True)
tokenizer = BertTokenizerFast.from_pretrained(tokenizer_path)

# Cihaza taşıma kodu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Test verisini token'lama
encodings = tokenizer(list(test_df['text_clean']), truncation=True, padding=True, return_tensors="pt")
input_ids = encodings['input_ids']
attention_mask = encodings['attention_mask']
true_labels = test_df['label'].values

# DataLoader ile batch'e çevirme kodu
batch_size = 16
test_dataset = TensorDataset(input_ids, attention_mask)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# Tahmin ve olasılıklar (batch'li)
all_preds = []
all_probs = []

with torch.no_grad():
    for batch in test_loader:
        b_input_ids = batch[0].to(device)
        b_attention_mask = batch[1].to(device)

        outputs = model(b_input_ids, attention_mask=b_attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, axis=1).cpu().numpy()
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

        all_preds.extend(preds)
        all_probs.extend(probs)

all_preds = np.array(all_preds)
all_probs = np.array(all_probs)

# Confusion Matrix Kodu
cm = confusion_matrix(true_labels, all_preds)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negatif', 'Pozitif'],
            yticklabels=['Negatif', 'Pozitif'])
plt.xlabel("Tahmin")
plt.ylabel("Gerçek")
plt.title("Confusion Matrix")
plt.show()

# ROC Curve Kodu
fpr, tpr, _ = roc_curve(true_labels, all_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.show()

# Precision-Recall Curve Kodu
precision, recall, _ = precision_recall_curve(true_labels, all_probs)

plt.figure(figsize=(6,5))
plt.plot(recall, precision, label="Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.show()

# Classification Report kodu
report = classification_report(true_labels, all_preds, digits=4)
print("📄 Classification Report:\n", report)
